# Statistical analysis of rainfall model variables

This notebook is dedicated to the statistical analysis of rainfall analysis

Listing files

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
RAINFALL_PATH = "C:\\Users\\ilang\\OneDrive\\Documentos\\Ilan\\academia\\dissertação\\data\\output"

In [6]:
rainfall_files = [f for f in os.listdir(RAINFALL_PATH) if f.endswith('train.csv')]
rainfall_files = [f for f in rainfall_files if not f.startswith('RIYADH')][:-2]
rainfall_files

['BELO HORIZONTE_train.csv',
 'CRUZEIRO DO SUL (ACRE)_train.csv',
 'DARWIN AIRPORT_train.csv',
 'GARANHUNS (PERNAMBUCO)_train.csv',
 'MANAUS_train.csv',
 'SALVADOR_train.csv']

Reading files

In [7]:
rainfall_dfs = [pd.read_csv(os.path.join(RAINFALL_PATH, f)) for f in rainfall_files]

In [22]:
def convert_date(df, date_column):
    df[date_column] = pd.to_datetime(df[date_column], format='%Y-%m-%d')
    return df

In [28]:
rainfall_dfs = [convert_date(df, 'Data Medicao') if 'Data Medicao' in df.columns else convert_date(df, 'date') for df in rainfall_dfs]

In [14]:
rainfall_data = [df['PRECIPITACAO TOTAL, DIARIO(mm)'].values for df in rainfall_dfs]

Reading ERA5 variables

In [16]:
ERA5_PATH = "data/input/ERA5"

Air temperature

In [17]:
air_temperature_files = [
    f for f in os.listdir(
        os.path.join(ERA5_PATH, "temperature")
        ) if f.endswith('.csv')][:-2]
air_temperature_files

['BELO_HORIZONTE_ERA5_temperature_2m_daily_2006_2025.csv',
 'CRUZEIRO_DO_SUL_ERA5_temperature_2m_daily_2006_2025.csv',
 'DARWIN_AIRPORT_ERA5_temperature_2m_daily_2006_2025.csv',
 'GARANHUNS_ERA5_temperature_2m_daily_2006_2025.csv',
 'MANAUS_ERA5_temperature_2m_daily_2006_2025.csv',
 'SALVADOR_ERA5_temperature_2m_daily_2006_2025.csv']

In [18]:
air_temperature_dfs = [pd.read_csv(os.path.join(ERA5_PATH, "temperature", f)) for f in air_temperature_files]

In [32]:
air_temperature_dfs = [convert_date(df, 'date') for df in air_temperature_dfs]

In [38]:
air_temperature_dfs = [
    df.merge(
        rainfall_dfs[i][['date', 'PRECIPITACAO TOTAL, DIARIO(mm)']], on='date', how='inner'
        ) if 'date' in rainfall_dfs[i].columns else df.merge(
            rainfall_dfs[i][['Data Medicao', 'PRECIPITACAO TOTAL, DIARIO(mm)']], left_on='date', right_on='Data Medicao', how='inner'
        ) for i, df in enumerate(air_temperature_dfs)]

Dew point temperature

In [41]:
dpt_files = [
    f for f in os.listdir(
        os.path.join(ERA5_PATH, "humidity")
        ) if f.endswith('.csv')][:-2]
dpt_files

['BELO_HORIZONTE_ERA5_dewpoint_2m_daily_2006_2025.csv',
 'CRUZEIRO_DO_SUL_ERA5_dewpoint_2m_daily_2006_2025.csv',
 'DARWIN_AIRPORT_ERA5_dewpoint_2m_daily_2006_2025.csv',
 'GARANHUNS_ERA5_dewpoint_2m_daily_2006_2025.csv',
 'MANAUS_ERA5_dewpoint_2m_daily_2006_2025.csv',
 'SALVADOR_ERA5_dewpoint_2m_daily_2006_2025.csv']

In [50]:
dpt_dfs = [pd.read_csv(os.path.join(ERA5_PATH, "humidity", f)) for f in dpt_files]

In [51]:
dpt_dfs = [convert_date(df, 'date') for df in dpt_dfs]

In [52]:
dpt_dfs = [
    df.merge(
        air_temperature_dfs[i][['date', 'PRECIPITACAO TOTAL, DIARIO(mm)', 'temperature_2m_c']], on='date', how='inner'
        )
    for i, df in enumerate(dpt_dfs)
]

Unified features files

In [56]:
data_matrix = [df[['date', 'PRECIPITACAO TOTAL, DIARIO(mm)', 'temperature_2m_c', 'dewpoint_2m_c']] for df in dpt_dfs]

In [59]:
data_matrix = [
    df.rename(
        columns={
            'PRECIPITACAO TOTAL, DIARIO(mm)': 'rainfall', 
            'temperature_2m_c': 'temperature', 
            'dewpoint_2m_c': 'dewpoint'
            }
        ) for df in data_matrix
    ]

In [65]:
data_matrix_lagged = [
    df.assign(
        rainfall_lagged=df['rainfall'].shift(1),
        temperature_lagged=df['temperature'].shift(1),
        dewpoint_lagged=df['dewpoint'].shift(1)
        ) for df in data_matrix
]

Obtaining correlation

In [70]:
for i, df in enumerate(data_matrix_lagged):
    print(os.listdir(os.path.join(ERA5_PATH, "humidity"))[i].split('_ERA5_dewpoint')[0].replace('_', ' ').title())
    print(
        data_matrix_lagged[i][[
            'rainfall', 'temperature_lagged', 'dewpoint_lagged', 'rainfall_lagged'
            ]].corr(method='pearson').to_latex(float_format="{:.2f}".format)
    )

Belo Horizonte
\begin{tabular}{lrrrr}
\toprule
 & rainfall & temperature_lagged & dewpoint_lagged & rainfall_lagged \\
\midrule
rainfall & 1.00 & 0.10 & 0.34 & 0.34 \\
temperature_lagged & 0.10 & 1.00 & 0.65 & 0.05 \\
dewpoint_lagged & 0.34 & 0.65 & 1.00 & 0.34 \\
rainfall_lagged & 0.34 & 0.05 & 0.34 & 1.00 \\
\bottomrule
\end{tabular}

Cruzeiro Do Sul
\begin{tabular}{lrrrr}
\toprule
 & rainfall & temperature_lagged & dewpoint_lagged & rainfall_lagged \\
\midrule
rainfall & 1.00 & -0.05 & 0.17 & 0.05 \\
temperature_lagged & -0.05 & 1.00 & 0.54 & -0.21 \\
dewpoint_lagged & 0.17 & 0.54 & 1.00 & 0.06 \\
rainfall_lagged & 0.05 & -0.21 & 0.06 & 1.00 \\
\bottomrule
\end{tabular}

Darwin Airport
\begin{tabular}{lrrrr}
\toprule
 & rainfall & temperature_lagged & dewpoint_lagged & rainfall_lagged \\
\midrule
rainfall & 1.00 & -0.03 & 0.25 & 0.36 \\
temperature_lagged & -0.03 & 1.00 & 0.74 & -0.01 \\
dewpoint_lagged & 0.25 & 0.74 & 1.00 & 0.25 \\
rainfall_lagged & 0.36 & -0.01 & 0.25 & 1.00 \\
\

ENSO

In [71]:
ENSO_PATH = "data/processed/pacific"

In [99]:
enso = pd.read_csv(os.path.join(ENSO_PATH, "ENSO_clean.csv"))
enso = convert_date(enso_daily, 'date')
enso = enso[['date', 'el_nino', 'la_nina', 'neutral']]
enso.index = enso['date']

Forward fill

In [100]:
enso = enso.resample('D').ffill().drop('date', axis=1).reset_index()

In [101]:
enso['el_nino_shifted'] = enso['el_nino'].shift(1)
enso['la_nina_shifted'] = enso['la_nina'].shift(1)
enso['neutral_shifted'] = enso['neutral'].shift(1)

Merge with data_matrix

In [112]:
data_matrix_enso = [
    df.merge(
        enso, on='date', how='inner'
    ) for df in data_matrix
]

In [113]:
def classify_enso(row):
    if row['el_nino_shifted'] == 1:
        return 'El Nino'
    elif row['la_nina_shifted'] == 1:
        return 'La Nina'
    elif row['neutral_shifted'] == 1:
        return 'Neutral'
    else:
        return 'Unknown'

In [114]:
for df in data_matrix_enso:
    df['enso_regime'] = df.apply(classify_enso, axis=1)

In [139]:
for i, df in enumerate(data_matrix_enso):
    
    loc = os.listdir(os.path.join(ERA5_PATH, "humidity"))[i].split('_ERA5_dewpoint')[0].replace('_', ' ').title()
    
    print("""
          \\begin{table}[H]
\\centering
\\footnotesize
\\caption{Variations in rainfall on day $t$ by ENSO regime on day $t-1$ in """ + loc + """}
\\resizebox{\\textwidth}{!}{
          """)

    data_mean = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].mean().rename(columns={'rainfall': 'mean'})
    data_mean_pos = data_matrix_enso[i][data_matrix_enso[i]['rainfall'] > 0].groupby('enso_regime')[['rainfall']].mean().rename(columns={'rainfall': 'mean positive'})
    data_max = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].max().rename(columns={'rainfall': 'max'})
    data_95 = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].quantile(0.95).rename(columns={'rainfall': 'q0.95'})
    data_99 = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].quantile(0.99).rename(columns={'rainfall': 'q0.99'})
    data_90 = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].quantile(0.90).rename(columns={'rainfall': 'q0.90'})
    data_75 = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].quantile(0.75).rename(columns={'rainfall': 'q0.75'})
    data_50 = data_matrix_enso[i].groupby('enso_regime')[['rainfall']].quantile(0.50).rename(columns={'rainfall': 'q0.50'})
    
    data = pd.concat([data_mean, data_max, data_95, data_99, data_90, data_75, data_50, data_mean_pos], axis=1).to_latex(float_format="{:.2f}".format)
    
    print(data)
    
    print("""
          }
          \\end{table}""")


          \begin{table}[H]
\centering
\footnotesize
\caption{Variations in rainfall on day $t$ by ENSO regime on day $t-1$ in Belo Horizonte}
\resizebox{\textwidth}{!}{
          
\begin{tabular}{lrrrrrrrr}
\toprule
 & mean & max & q0.95 & q0.99 & q0.90 & q0.75 & q0.50 & mean positive \\
enso_regime &  &  &  &  &  &  &  &  \\
\midrule
El Nino & 4.78 & 171.80 & 28.48 & 59.89 & 15.50 & 2.20 & 0.00 & 14.61 \\
La Nina & 5.95 & 126.80 & 35.26 & 64.56 & 20.86 & 4.25 & 0.00 & 15.96 \\
Neutral & 1.58 & 84.60 & 11.01 & 34.47 & 2.40 & 0.00 & 0.00 & 10.91 \\
\bottomrule
\end{tabular}


          }
          \end{table}

          \begin{table}[H]
\centering
\footnotesize
\caption{Variations in rainfall on day $t$ by ENSO regime on day $t-1$ in Cruzeiro Do Sul}
\resizebox{\textwidth}{!}{
          
\begin{tabular}{lrrrrrrrr}
\toprule
 & mean & max & q0.95 & q0.99 & q0.90 & q0.75 & q0.50 & mean positive \\
enso_regime &  &  &  &  &  &  &  &  \\
\midrule
El Nino & 6.91 & 105.30 & 33.65 & 59.85 & 22